# 정적 웹페이지 수집하기
* 순수 HTML, CSS로 만들어진 페이지
* javascrip로 내용을 갱신하지 않는 페이지

# yes24 베스트셀러 자료 수집하기

In [1]:
import pandas as pd
import requests
import time
from bs4 import BeautifulSoup as bs

In [12]:
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=1&pageSize=24

SyntaxError: invalid syntax (1327645662.py, line 1)

In [ ]:
url = "https://www.yes24.com/product/category/bestseller"
payload = dict(categoryNumber="001", pageNumber=1, pageSize=120)
r = requests.get(url, params=payload)
print(r.url)
print(r.status_code)
soup = bs(r.content, "lxml")
time.sleep(5)
# soup

* 전체 yes24 베스트셀러 페이지중 책 정보가 들어있는 곳: ul#yesBestList
* ul#yesBestList 아래의 li에 책 1권의 정보가 들어있음.

In [125]:
# soup.select_one("ul#yesBestList").select("li")

In [19]:
# soup.select_one("ul#yesBestList")

In [20]:
# yes24 전체 페이지에서 책 정보가 들어있는 부분만 잘라서 book_list에 저장
book_list = soup.select("ul#yesBestList > li")

In [22]:
# book_list[:2]

In [24]:
result = {}
for idx, book in enumerate(book_list):
    print(f"{idx}/{len(book_list)} 추출중", end="\r")
    # 책 제목
    book_title = book.select_one(".gd_name").text
    # 저자
    author = book.select_one(".info_row.info_pubGrp a").text
    # 출판사
    publisher = book.select_one(".authPub.info_pub a").text
    # 출간일
    date_pub = book.select_one(".authPub.info_date").text
    # 가격
    price = book.select_one(".info_row.info_price em.yes_b").text
    # 평점
    rating = book.select_one(".rating_grade em.yes_b").text if book.select_one(".rating_grade em.yes_b") != None else 0.0
    # 리뷰수
    n_reviews = book.select_one(".info_row.info_rating em.txC_blue").text if book.select_one(".info_row.info_rating em.txC_blue") != None else 0
    
    keys = ['book_title', 'author', 'publisher', 'date_pub', 'price', 'rating', 'n_reviews']
    values = [book_title, author, publisher, date_pub, price, rating, n_reviews]
    for key, value in zip(keys, values):
        result.setdefault(key, []).append(value)

for key, value in result.items():
    print(key, len(value))
    
df = pd.DataFrame(result)
df

book_title 120
author 120
publisher 120
date_pub 120
price 120
rating 120
n_reviews 120


,book_title,author,publisher,date_pub,price,rating,n_reviews
0,단 한 번의 삶,김영하,복복서가,2025년 04월,"15,120",8.6,13
1,듀얼 브레인,이선 몰릭,상상스퀘어,2025년 03월,"18,900",8.7,38
2,어른의 품격을 채우는 100일 필사 노트,김종원,청림Life,2025년 03월,"18,000",9.9,56
3,소년이 온다,한강,창비,2014년 05월,"13,500",9.7,"3,848"
4,성적 초격차를 만드는 독서력 수업,김수미,빅피시,2025년 03월,"17,820",9.9,69
...,...,...,...,...,...,...,...
115,해커스 토익 RC Reading(리딩) 기본서,David Cho,해커스어학연구소,2023년 07월,"16,920",9.8,241
116,침묵의 퍼레이드,히가시노 게이고,재인,2025년 03월,"19,620",10.0,10
117,2025 시대에듀 투자자산운용사 실제유형 모의고사 + 특별부록 PASSCODE Pr...,유창호,시대고시기획 시대교육,2025년 03월,"49,500",10.0,2
118,마흔 고비에 꼭 만나야 할 장자,이길환,이든서재,2025년 04월,"16,920",0.0,0


In [27]:
x = 1234878
print("짝수") if x % 2 == 0 else print("홀수")
    

짝수


In [23]:
print("가격: ", len(list(soup.select("ul#yesBestList .info_row.info_price em.yes_b"))))
print("평점: ", len(list(soup.select("ul#yesBestList .rating_grade em.yes_b"))))
print("리뷰수: ", len(list(soup.select("ul#yesBestList .info_row.info_rating em.txC_blue"))))

가격:  120
평점:  106
리뷰수:  106


In [30]:
result['author'][29]

'정관 스님'

In [33]:
book_list[29].select_one(".authPub.info_auth").text

'\n정관 스님, 후남 셀만 저/베로니크 회거 사진/양혜영 역\r\n                            '

In [34]:
book_list[29].select_one(".authPub.info_auth").text.split("/")

['\n정관 스님, 후남 셀만 저', '베로니크 회거 사진', '양혜영 역\r\n                            ']

In [151]:
book_list[28].select_one(".authPub.info_auth").text.split("/")[1]

'베로니크 회거 사진'

In [152]:
book_list[28].select_one(".authPub.info_auth").text.split("/")[2]

'양혜영 역\r\n                            '

In [40]:
book_list[18].select_one(".authPub.info_auth").text.split("/")[1].strip("감추기\n ").replace("\n", " ")

'백온유 강보라 서장원 성해나 성혜령 이희주 현호정'

In [ ]:
https://www.yes24.com/product/goods/101402

In [170]:
link = "https://www.yes24.com" + book_list[20].select_one(".gd_name")['href']

'https://www.yes24.com/product/goods/144248311'

# 전체 페이지 수집하기

In [165]:
import pandas as pd
import requests
import time
from bs4 import BeautifulSoup as bs

In [6]:
result_list = []
page = 1
while True:
    url = "https://www.yes24.com/product/category/bestseller"
    payload = dict(categoryNumber="001", pageNumber=page, pageSize=120)
    r = requests.get(url, params=payload)
    print(r.url)
    print(r.status_code)
    soup = bs(r.content, "lxml")
    time.sleep(5)
    # yes24 전체 페이지에서 책 정보가 들어있는 부분만 잘라서 book_list에 저장
    book_list = soup.select("ul#yesBestList > li")
    result = {}
    for idx, book in enumerate(book_list):
        print(f"{idx}/{len(book_list)} 추출중", end="\r")
        # 책 제목
        book_title = book.select_one(".gd_name").text
        # 저자
        author, photo, trans, paint = author_extraction(book)
        # 출판사
        publisher = book.select_one(".authPub.info_pub a").text
        # 출간일
        date_pub = book.select_one(".authPub.info_date").text
        # 가격
        price = book.select_one(".info_row.info_price em.yes_b").text
        # 평점
        rating = book.select_one(".rating_grade em.yes_b").text if book.select_one(".rating_grade em.yes_b") != None else 0.0
        # 리뷰수
        n_reviews = book.select_one(".info_row.info_rating em.txC_blue").text if book.select_one(".info_row.info_rating em.txC_blue") != None else 0

        keys = ['book_title', 'author', 'photo', 'trans', 'paint', 'publisher', 'date_pub', 'price', 'rating', 'n_reviews']
        values = [book_title, author, photo, trans, paint, publisher, date_pub, price, rating, n_reviews]
        for key, value in zip(keys, values):
            result.setdefault(key, []).append(value)

    for key, value in result.items():
        print(key, len(value))

    result_list.append(pd.DataFrame(result))
    
    if page < 10:
        page += 1
    else:
        break
    
result_list = pd.concat(result_list)
result_list = result_list.reset_index(drop=True)
result_list

https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=1&pageSize=120
200
book_title 120
author 120
photo 120
trans 120
paint 120
publisher 120
date_pub 120
price 120
rating 120
n_reviews 120
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=2&pageSize=120
200
book_title 120
author 120
photo 120
trans 120
paint 120
publisher 120
date_pub 120
price 120
rating 120
n_reviews 120
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=3&pageSize=120
200
book_title 120
author 120
photo 120
trans 120
paint 120
publisher 120
date_pub 120
price 120
rating 120
n_reviews 120
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=4&pageSize=120
200
book_title 120
author 120
photo 120
trans 120
paint 120
publisher 120
date_pub 120
price 120
rating 120
n_reviews 120
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=5&pageSize=120
200
book_title 120
author 120
photo 1

,book_title,author,photo,trans,paint,publisher,date_pub,price,rating,n_reviews
0,단 한 번의 삶,김영하 저,,,,복복서가,2025년 04월,"15,120",8.6,13
1,듀얼 브레인,이선 몰릭 저,,신동숙 역,,상상스퀘어,2025년 03월,"18,900",8.7,38
2,어른의 품격을 채우는 100일 필사 노트,김종원 저,,,,청림Life,2025년 03월,"18,000",9.9,56
3,소년이 온다,한강 저,,,,창비,2014년 05월,"13,500",9.7,"3,848"
4,성적 초격차를 만드는 독서력 수업,김수미 저,,,,빅피시,2025년 03월,"17,820",9.9,69
...,...,...,...,...,...,...,...,...,...,...
994,태도에 관하여 (20만 부 기념 완결판),임경선 저,,,,토스트,2024년 09월,"16,200",9.3,33
995,장수탕 선녀님,,,,백희나 글그림,스토리보울,2024년 04월,"13,500",9.8,24
996,중등 필독 신문,"이현옥, 이현주 저",,,,체인지업,2024년 02월,"16,020",9.9,156
997,[예스리커버] 나는 나의 스무 살을 가장 존중한다,이하영 저,,,,토네이도,2024년 08월,"16,200",9.4,212


# 저자, 역자, 글그림, 편저, 공동저자등 구분하기

In [2]:
def text_clean(text):
    return text.replace("\n", " ").replace("\r", " ").strip()

In [3]:
def author_extraction(book):
    author = ""
    photo = ""
    trans = ""
    paint = ""
    for idx, item in enumerate(text_clean(book.select_one(".authPub.info_auth").text).split("/")):
        print(idx, item)
        if idx == 0:
            if "저" == item[-1]:
                author = text_clean(item[:-2])
            elif "글" == item[-1]:
                author = text_clean(item[:-2])
            elif "글그림" == item[-3:]:
                author = text_clean(item[:-4])
        else:        
            if "정보 더 보기" == item[-7:]:
                author = text_clean(item.replace("정보 더 보기", ""))
            elif "사진" == item[-2:]:
                photo = text_clean(item[:-3])
            elif "역" == item[-1]:
                trans = text_clean(item[:-2])
            elif "글그림" == item[-3:]:
                paint = text_clean(item[:-4])
            
    print(f"author{author}, photo{photo}, trans{trans}, paint{paint}")
    return author, photo, trans, paint

In [4]:
url = "https://www.yes24.com/product/category/bestseller"
payload = dict(categoryNumber="001", pageNumber=1, pageSize=120)
r = requests.get(url, params=payload)
print(r.url)
print(r.status_code)
soup = bs(r.content, "lxml")
time.sleep(5)
book_list = soup.select("ul#yesBestList > li")
for book in book_list[:2]:
    author, photo, trans, paint = author_extraction(book)
    print(author, photo, trans, paint)

https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=1&pageSize=120
200
0 김영하 저
author김영하, photo, trans, paint
김영하   
0 이선 몰릭 저
1 신동숙 역
author이선 몰릭, photo, trans신동숙, paint
이선 몰릭  신동숙 


In [203]:
for item in text_clean(book_list[20].select_one(".authPub.info_auth").text).split("/"):
#     print(item)
    if "감추기" in item[:4]:
        author = item.replace("감추기", "").strip()
        print(author)

백온유 강보라 서장원 성해나 성혜령 이희주 현호정


# 저자, 역자 세분화 코드 적용